In [0]:
storage_account = "stfintechpipeline"

storage_account_key = "YOUR_ACCESS_KEY_HERE"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)

CLEANED = f"abfss://cleaned@{storage_account}.dfs.core.windows.net"
CURATED = f"abfss://curated@{storage_account}.dfs.core.windows.net"

print("Connection configured.")
print(f"Reading from : {CLEANED}")
print(f"Writing to   : {CURATED}")

Connection configured.
Reading from : abfss://cleaned@stfintechpipeline.dfs.core.windows.net
Writing to   : abfss://curated@stfintechpipeline.dfs.core.windows.net


In [0]:
#load the cleaned parquet files
df_transactions = spark.read.parquet(f"{CLEANED}/transactions_data")
df_users = spark.read.parquet(f"{CLEANED}/users_data")
df_cards = spark.read.parquet(f"{CLEANED}/cards_data")

print(f"transactions : {df_transactions.count():,} rows")
print(f"cards        : {df_cards.count():,} rows")
print(f"users        : {df_users.count():,} rows")


transactions : 13,305,915 rows
cards        : 6,146 rows
users        : 2,000 rows


In [0]:
from pyspark.sql.functions import max as spark_max, min as spark_min
df_transactions.select(
    spark_min("transaction_date").alias("earliest"),
    spark_max("transaction_date").alias("latest")
).show()

+-------------------+-------------------+
|           earliest|             latest|
+-------------------+-------------------+
|2010-01-01 00:01:00|2019-10-31 23:59:00|
+-------------------+-------------------+



In [0]:
#imports and constants
#dataset_end is the last date in the dataset
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, datediff, lit, when,
    round as spark_round, to_date, max as spark_max,
    min as spark_min, hour, dayofweek, month, year,
    dayofmonth, date_format, first, concat, abs as spark_abs,
    sqrt, pow as spark_pow
)
from pyspark.sql.types import IntegerType, DoubleType
import pyspark.sql.functions as F

DATASET_END = to_date(lit("2019-10-31"))

add time based columns to transactions table
used in dim_time for star schema and as inputs for customer-level aggregations


In [0]:
#extract date parts
df_transactions = df_transactions \
  .withColumn("transaction_hour", hour(col("transaction_date")))\
  .withColumn("transaction_month", month(col("transaction_date")))\
  .withColumn("transaction_year", year(col("transaction_date")))\
  .withColumn("transaction_day", dayofmonth(col("transaction_date")))\
  .withColumn("day_of_week", date_format(col("transaction_date"), "EEEE"))\
  .withColumn("is_weekend", 
              when(dayofweek(col("transaction_date")).isin([1,7]), True)
              .otherwise(False))

print("time features:")
df_transactions.select("transaction_date","transaction_hour","day_of_week","is_weekend").show(5)

time features:
+-------------------+----------------+-----------+----------+
|   transaction_date|transaction_hour|day_of_week|is_weekend|
+-------------------+----------------+-----------+----------+
|2010-01-01 00:27:00|               0|     Friday|     false|
|2010-01-01 00:36:00|               0|     Friday|     false|
|2010-01-01 00:57:00|               0|     Friday|     false|
|2010-01-01 01:01:00|               1|     Friday|     false|
|2010-01-01 01:19:00|               1|     Friday|     false|
+-------------------+----------------+-----------+----------+
only showing top 5 rows



customer-level aggregations

In [0]:
#total_transaction_count and total_spend
#total_spend sums amount_clean- negatives(refunds) reduce the total, which is the correct behavior

agg_base = df_transactions.groupBy("client_id").agg(
  count("id").alias("total_transaction_count"),
  spark_round(spark_sum("amount_clean"),2).alias("total_spend"),
  spark_round(avg("amount_clean"),2).alias("avg_spend_per_transaction"),
  spark_max("transaction_date").alias("last_transaction_date"),
  spark_min("transaction_date").alias("first_transaction_date")
)

print(f"Customers aggregated: {agg_base.count():,}")
agg_base.show(5)

Customers aggregated: 1,219
+---------+-----------------------+-----------+-------------------------+---------------------+----------------------+
|client_id|total_transaction_count|total_spend|avg_spend_per_transaction|last_transaction_date|first_transaction_date|
+---------+-----------------------+-----------+-------------------------+---------------------+----------------------+
|     1591|                  21888|  620499.95|                    28.35|  2019-10-31 11:45:00|   2010-01-01 08:02:00|
|     1645|                   9259|  353480.74|                    38.18|  2019-10-31 09:57:00|   2010-01-01 09:36:00|
|     1959|                   5190|  444991.17|                    85.74|  2019-10-31 01:19:00|   2010-01-01 12:00:00|
|      148|                  11101|  389935.97|                    35.13|  2019-10-31 11:04:00|   2010-01-01 07:21:00|
|      496|                   5636|  240852.55|                    42.73|  2019-10-29 15:39:00|   2010-01-01 15:00:00|
+---------+---------

In [0]:
#days since last transaction-meausres how long ago each customer last transacted.high values could indicate disengaged customer
agg_base = agg_base.withColumn(
    "days_since_last_transaction",
    datediff(DATASET_END, to_date(col("last_transaction_date"))))
print("days_since_last_transaction sample:")
agg_base.select("client_id", "last_transaction_date",
                "days_since_last_transaction").show(5)


days_since_last_transaction sample:
+---------+---------------------+---------------------------+
|client_id|last_transaction_date|days_since_last_transaction|
+---------+---------------------+---------------------------+
|     1591|  2019-10-31 11:45:00|                          0|
|     1645|  2019-10-31 09:57:00|                          0|
|     1959|  2019-10-31 01:19:00|                          0|
|      148|  2019-10-31 11:04:00|                          0|
|      496|  2019-10-29 15:39:00|                          2|
+---------+---------------------+---------------------------+
only showing top 5 rows



In [0]:
#avg-days between transactions-average gap in days between consecutive transactions per customer. High values = low engagement = churn signal
# Calculated as total active days / total transactions
agg_base = agg_base.withColumn(
    "avg_days_between_transactions",
    spark_round(
        datediff(
            to_date(col("last_transaction_date")),
            to_date(col("first_transaction_date"))
        )/col("total_transaction_count"),2
    )
)
print("avg_days_between_transactions sample:")
agg_base.select("client_id", "total_transaction_count",
                "avg_days_between_transactions").show(5)

avg_days_between_transactions sample:
+---------+-----------------------+-----------------------------+
|client_id|total_transaction_count|avg_days_between_transactions|
+---------+-----------------------+-----------------------------+
|     1591|                  21888|                         0.16|
|     1645|                   9259|                         0.39|
|     1959|                   5190|                         0.69|
|      148|                  11101|                         0.32|
|      496|                   5636|                         0.64|
+---------+-----------------------+-----------------------------+
only showing top 5 rows



In [0]:
#merchant behavior features
#unique merchant count:declining=disengagement signal
#unique merchant category count-narrow spend- churn signal
# top merchant category- dominant category for segmentation

agg_merchant = df_transactions.groupby("client_id").agg(
    F.countDistinct("merchant_id").alias("unique_merchant_count"),
    F.countDistinct("merchant_category").alias("unique_merchant_category_count"),
    F.max("merchant_category").alias("top_merchant_category")
)
print("merchant features sample:")
agg_merchant.show(5)


merchant features sample:
+---------+---------------------+------------------------------+---------------------+
|client_id|unique_merchant_count|unique_merchant_category_count|top_merchant_category|
+---------+---------------------+------------------------------+---------------------+
|     1238|                  457|                            87|      Wholesale Clubs|
|     1645|                  327|                            83| Women's Ready-To-...|
|     1959|                  278|                            86| Women's Ready-To-...|
|      496|                  219|                            76| Women's Ready-To-...|
|     1591|                  375|                            82|      Wholesale Clubs|
+---------+---------------------+------------------------------+---------------------+
only showing top 5 rows



In [0]:
#error count-total transactions with any error
#error rate- proportion of transactions with errors
#insufficient balance count- count for this error type as it signals financial stress

agg_fraud_errors = df_transactions\
    .withColumn("has_error",
                when(col("errors")!= "No Error",1).otherwise(0))\
    .withColumn("has_insufficienct_balance",
                when(col("errors").contains("Insufficient Balance"), 1).otherwise(0))\
    .withColumn("is_fraud_int", when(col("is_fraud") == "Yes", 1).otherwise(0))\
    .groupBy("client_id").agg(
        spark_sum("has_error").alias("error_count"),
        spark_round(spark_sum("has_error")/count("id"),4).alias("error_rate"),
        spark_sum("has_insufficienct_balance").alias("insufficient_balance_count"),
        spark_round(spark_sum("is_fraud_int")/count("id"), 6).alias("pct_fraud_transactions"),
        when(spark_sum("is_fraud_int")>0, True)
            .otherwise(False).alias("has_fraud_history"))
    
print("Error and fraud features sample:")
agg_fraud_errors.show(5)

print("\nFraud history distribution:")
agg_fraud_errors.groupBy("has_fraud_history").count().show()

Error and fraud features sample:
+---------+-----------+----------+--------------------------+----------------------+-----------------+
|client_id|error_count|error_rate|insufficient_balance_count|pct_fraud_transactions|has_fraud_history|
+---------+-----------+----------+--------------------------+----------------------+-----------------+
|     1591|        338|    0.0154|                       122|               5.48E-4|             true|
|     1645|        125|    0.0135|                        63|               2.16E-4|             true|
|     1959|        107|    0.0206|                        83|              0.001734|             true|
|      148|        150|    0.0135|                        91|              0.001171|             true|
|      496|         86|    0.0153|                        48|              0.004968|             true|
+---------+-----------+----------+--------------------------+----------------------+-----------------+
only showing top 5 rows


Fraud history 

In [0]:
#transaction frequency trend-Compares each customer's transaction count in the last 90 days of the dataset vs their historical daily average
#Ratio < 1 means slowing down = churn signal
# Ratio > 1 means accelerating = healthy engagement

TREND_START = to_date(lit("2019-08-02"))
recent = df_transactions\
    .filter(to_date(col("transaction_date"))>=TREND_START)\
    .groupBy("client_id")\
    .agg(count("id").alias("recent_90d_count"))

#historical daily rate = total transcation/total active days
historical = agg_base.select(
    "client_id",
    "total_transaction_count",
    "first_transaction_date",
    "last_transaction_date"
).withColumn(
    "active_days",
    datediff(
        to_date(col("last_transaction_date")),
        to_date(col("first_transaction_date"))
    )+ 1
).withColumn(
    "historical_daily_rate",
    col("total_transaction_count")/col("active_days")
).withColumn(
    "expected_90d_count",
    spark_round(col("historical_daily_rate")*90,2)
)

#join and calculate trend ratio
agg_trend = historical \
    .join(recent, on="client_id", how="left")\
    .withColumn("recent_90d_count",
                when(col("recent_90d_count").isNull(),0)
                .otherwise(col("recent_90d_count")))\
    .withColumn("transaction_frequency_trend",
                spark_round(col("recent_90d_count")/col("expected_90d_count"),4))\
    .select("client_id","transaction_frequency_trend")

print("transaction_frequency_trend sample:")
agg_trend.show(5)


transaction_frequency_trend sample:
+---------+---------------------------+
|client_id|transaction_frequency_trend|
+---------+---------------------------+
|     1591|                     1.0701|
|     1645|                     1.1204|
|     1959|                     1.0763|
|      148|                     0.9956|
|      496|                      1.026|
+---------+---------------------------+
only showing top 5 rows



In [0]:
#time pattern features
#pct weekend transactions-behavioral signal
#online transactions-channel preference
#most common hour- behavioral fingerprint+fraud context

agg_time = df_transactions.groupBy("client_id").agg(
    spark_round(
        spark_sum(when(col("is_weekend"),1).otherwise(0))/count("id"),4
    ).alias("pct_weekend_transactions"),
    spark_round(
        spark_sum(when(col("use_chip")== "Online Transaction", 1).otherwise(0))/count("id"),4
    ).alias("pct_online_transactions"),
    first(col("transaction_hour"),ignorenulls=True).alias("most_common_hour")
)

print("Time pattern features sample:")
agg_time.show(5)


Time pattern features sample:
+---------+------------------------+-----------------------+----------------+
|client_id|pct_weekend_transactions|pct_online_transactions|most_common_hour|
+---------+------------------------+-----------------------+----------------+
|     1591|                  0.2916|                 0.4806|               8|
|     1645|                  0.2754|                   0.06|               9|
|     1959|                  0.2958|                 0.1108|              13|
|      148|                  0.2839|                 0.0733|               8|
|      496|                   0.285|                 0.0436|              15|
+---------+------------------------+-----------------------+----------------+
only showing top 5 rows



In [0]:
#location feature
#home state transactions-proportion of transactions
#in customer's home state- required joining users to get home state derived from latitude to longitude via a state lookup or approximated using the most frequrent merchant state
#most frequent merchant state is used as home state proxy since we dont have a direct home state column

from pyspark.sql import Window
#step 1: derive each customer's most frequent transaction state as proxy for home state
state_counts = df_transactions\
    .filter(col("merchant_state")!= "Online")\
    .groupBy("client_id", "merchant_state")\
    .agg(count("id").alias("state_count"))

w= Window.partitionBy("client_id").orderBy(col("state_count").desc())
home_state = state_counts\
    .withColumn("rank", F.rank().over(w))\
    .filter(col("rank")==1)\
    .select(col("client_id"),col("merchant_state").alias("home_state"))

#step 2-join back and calculate proportion
agg_location = df_transactions\
    .join(home_state, on="client_id", how="left")\
    .withColumn("is_home_state",
                when(col("merchant_state")== col("home_state"),1).otherwise(0))\
    .groupBy("client_id").agg(
        spark_round(spark_sum("is_home_state")/count("id"),4).alias("pct_home_state_transactions")
    )

print("location feature sample")
agg_location.show(5)


location feature sample
+---------+---------------------------+
|client_id|pct_home_state_transactions|
+---------+---------------------------+
|     1591|                     0.4602|
|     1645|                     0.8446|
|     1959|                     0.7829|
|      148|                     0.8897|
|      496|                     0.8719|
+---------+---------------------------+
only showing top 5 rows



Derived columns- calculated directly from users and cards data

In [0]:
#card level derived features
#has expired card- card expiry before dataset end=disengagement
#daya until card expiry-urgency signal
#has both card type-product breadth
#pin change recency-account engagement proxy
#account tenure days
#days between account open date and last transaction date
#longer tenure=less churn
#since acct_open_date is stored as MM/YYYY-parse to first of month
df_cards = df_cards\
    .withColumn("card_expiry_parsed",
                to_date(concat(lit("01/"),col("expires")),"dd/MM/yyyy"))\
    .withColumn("has_expired_card",
                when(col("card_expiry_parsed")< DATASET_END, True).otherwise(False))\
    .withColumn("days_until_card_expiry",
                datediff(col("card_expiry_parsed"), DATASET_END))\
    .withColumn("acct_open_parsed",
                to_date(concat(lit("01/"),col("acct_open_date")), "dd/MM/yyyy"))\
    .withColumn("account_tenure_days",
                datediff(DATASET_END, col("acct_open_parsed")))\
    .withColumn("pin_change_recency",
                2019-col("year_pin_last_changed"))\
    .drop("card_expiry_parsed", "acct_open_parsed")

#aggregate to customer level
card_features = df_cards.groupby("client_id").agg(
    spark_max("spending_limit").alias("max_spending_limit"),
    spark_max("account_tenure_days").alias("max_account_tenure_days"),
    spark_max("days_until_card_expiry").alias("days_until_card_expiry"),
    spark_sum(when(col("has_chip")== "YES", 1).otherwise(0)).alias("chip_card_count"),
    spark_sum(when(col("has_expired_card"),1).otherwise(0)).alias("has_expired_card"),
    F.countDistinct("card_type").alias("distinct_card_types"),
    spark_min("pin_change_recency").alias("min_pin_change_recency")
)


In [0]:
#spending limit utilization
#average transaction amount as a proportion of spending limit
#high utilixzation=customer is maxing out their card=financial stress=churn signal

avg_spend = agg_base.select("client_id", "avg_spend_per_transaction")

card_features = card_features\
    .join(avg_spend,on="client_id", how="left")\
    .withColumn("spending_limit_utilization",
                spark_round(
                  when(col("avg_spend_per_transaction").isNull(), lit(0))
                .otherwise(col("avg_spend_per_transaction")) /
                 when(col("max_spending_limit") == 0, lit(1))
                 .otherwise(col("max_spending_limit")), 4
        )) \
    .withColumn("has_both_card_types",
                when(col("distinct_card_types")>1, True).otherwise(False))\
    .drop("avg_spend_per_transaction", "distinct_card_types")
    
print("Card features sample:")
card_features.show(5)


Card features sample:
+---------+------------------+-----------------------+----------------------+---------------+----------------+----------------------+--------------------------+-------------------+
|client_id|max_spending_limit|max_account_tenure_days|days_until_card_expiry|chip_card_count|has_expired_card|min_pin_change_recency|spending_limit_utilization|has_both_card_types|
+---------+------------------+-----------------------+----------------------+---------------+----------------+----------------------+--------------------------+-------------------+
|     1088|           15194.0|                   5173|                  1158|              3|               1|                     6|                    0.0027|               true|
|     1591|           17332.0|                   6088|                  1858|              6|               1|                     0|                    0.0016|               true|
|      496|           22516.0|                   6269|                  1

In [0]:
#User-level derived features
#debt to income ratio- high values signal financial stress 
# years to retirement- customers near retirement may change spending behavior
#income area ratio, cards per institution

df_users = df_users \
  .withColumn("debt_to_income_ratio",
              spark_round(
                col("total_debt")/when(col("yearly_income")== 0, lit(1))
                .otherwise(col("yearly_income")), 4
              ))\
  .withColumn("years_to_retirement",
              col("retirement_age")- col("current_age"))\
  .withColumn("income_to_area_ratio",
              spark_round(
                col("yearly_income")/
                when(col("per_capita_income")==0,lit(1))
                .otherwise(col("per_capita_income")),4
              ))

print("derived user features:")
df_users.select("id", "debt_to_income_ratio", "years_to_retirement", "income_to_area_ratio").show(5)


derived user features:
+----+--------------------+-------------------+--------------------+
|  id|debt_to_income_ratio|years_to_retirement|income_to_area_ratio|
+----+--------------------+-------------------+--------------------+
| 825|              2.1377|                 13|              2.0389|
|1746|              2.4769|                 15|              2.0388|
|1718|              0.0059|                -14|              1.4763|
| 708|              0.8096|                  0|              1.5319|
|1164|              1.6762|                 27|              2.0389|
+----+--------------------+-------------------+--------------------+
only showing top 5 rows



binned columns
convert continous variables into categorical bands

In [0]:
#debt_to_income_band
#based on EDA- low=<0.2, medium:0.2-0.5, high:0.5-1.0, very high:1+

df_users = df_users\
    .withColumn("credit_score_band",
                when(col("credit_score")<580, "Poor")
                .when(col("credit_score")<670, "Fair")
                .when(col("credit_score")<740, "Good")
                .otherwise("Excellent"))\
    .withColumn("age_group",
                when(col("current_age")<30, "Under 30")
                .when(col("current_age")<45, "30-44")
                .when(col("current_age")<60, "45-59")
                .otherwise("60+"))\
    .withColumn("debt_to_income_band",
                when(col("debt_to_income_ratio")<0.2, "Low")
                .when(col("debt_to_income_ratio")<0.5, "Medium")
                .when(col("debt_to_income_ratio")<1.0, "High")
                .otherwise("Very High"))

print("Binned columns distribution:")
df_users.groupBy("credit_score_band").count().orderBy("credit_score_band").show()
df_users.groupBy("age_group").count().orderBy("age_group").show()
df_users.groupBy("debt_to_income_band").count().orderBy("debt_to_income_band").show()

Binned columns distribution:
+-----------------+-----+
|credit_score_band|count|
+-----------------+-----+
|        Excellent|  640|
|             Fair|  348|
|             Good|  931|
|             Poor|   81|
+-----------------+-----+

+---------+-----+
|age_group|count|
+---------+-----+
|    30-44|  531|
|    45-59|  543|
|      60+|  446|
| Under 30|  480|
+---------+-----+

+-------------------+-----+
|debt_to_income_band|count|
+-------------------+-----+
|               High|  247|
|                Low|  268|
|             Medium|  144|
|          Very High| 1341|
+-------------------+-----+



Customer features table- one row per customer. feeds the churn model and customer level fraud model

In [0]:
#join all feature groups together

#user features
user_features = df_users.select(
  col("id").alias("client_id"),
    "current_age", "gender", "yearly_income", "total_debt", "per_capita_income", "credit_score", "num_credit_cards", "latitude", "longitude", "debt_to_income_ratio", "years_to_retirement",
    "income_to_area_ratio","credit_score_band", "age_group", "debt_to_income_band"
)

#assemble
customer_features = agg_base \
.select("client_id", "total_transaction_count", "total_spend", "avg_spend_per_transaction", "days_since_last_transaction", "avg_days_between_transactions") \
    .join(agg_merchant, on="client_id", how="left") \
    .join(agg_fraud_errors, on="client_id", how="left") \
    .join(agg_trend,on="client_id", how="left") \
    .join(agg_time,on="client_id", how="left") \
    .join(agg_location,on="client_id", how="left") \
    .join(card_features,on="client_id", how="left") \
    .join(user_features,on="client_id", how="left")
  
print(f"customer_features row count: {customer_features.count():,}")
print(f"customer_features columns: {len(customer_features.columns)}")
customer_features.printSchema()

customer_features row count: 1,219
customer_features columns: 42
root
 |-- client_id: integer (nullable = true)
 |-- total_transaction_count: long (nullable = false)
 |-- total_spend: double (nullable = true)
 |-- avg_spend_per_transaction: double (nullable = true)
 |-- days_since_last_transaction: integer (nullable = true)
 |-- avg_days_between_transactions: double (nullable = true)
 |-- unique_merchant_count: long (nullable = true)
 |-- unique_merchant_category_count: long (nullable = true)
 |-- top_merchant_category: string (nullable = true)
 |-- error_count: long (nullable = true)
 |-- error_rate: double (nullable = true)
 |-- insufficient_balance_count: long (nullable = true)
 |-- pct_fraud_transactions: double (nullable = true)
 |-- has_fraud_history: boolean (nullable = true)
 |-- transaction_frequency_trend: double (nullable = true)
 |-- pct_weekend_transactions: double (nullable = true)
 |-- pct_online_transactions: double (nullable = true)
 |-- most_common_hour: integer (null

assemble transaction features
One row per transaction. Feeds transaction-level fraud model
Joins customer context from customer_features onto each transaction row so the model sees both the transaction itself and the customer's profile at that point.

In [0]:
customer_context = customer_features.select(
  "client_id", "credit_score", "debt_to_income_ratio",
    "avg_spend_per_transaction", "error_rate",
    "pct_online_transactions", "max_account_tenure_days"
)

transaction_features = df_transactions\
  .join(customer_context, on="client_id", how="left")\
  .join(home_state, on="client_id",how="left")\
  .withColumn("amount_vs_customer_avg",
              spark_round(
                col("amount_clean")/
                when(col("avg_spend_per_transaction")== 0, lit(1))
                .otherwise(col("avg_spend_per_transaction")), 4
              ))\
  .withColumn("is_home_state",
              when(col("merchant_state")== col("home_state"), 1).otherwise(0))\
  .withColumn("is_high_risk_hour",
              when(col("transaction_hour").between(0,4),1).otherwise(0))\
  .withColumn("error_encoded",
              when(col("errors")!= "No Error",1).otherwise(0))\
  .withColumn("use_chip_encoded",
              when(col("use_chip")== "Swipe Transaction",0)
              .when(col("use_chip")== "Chip Transaction", 1)
              .otherwise(2))\
  .withColumn("is_fraud_label",
              when(col("is_fraud")== "Yes",1).otherwise(0))\
  .select(
     "id", "client_id", "amount_clean", "transaction_hour",
        "is_weekend", "use_chip_encoded", "merchant_state",
        "merchant_category", "error_encoded", "is_fraud_label",
        "credit_score", "debt_to_income_ratio",
        "avg_spend_per_transaction", "error_rate",
        "pct_online_transactions", "max_account_tenure_days",
        "amount_vs_customer_avg", "is_home_state",
        "is_high_risk_hour"
    )

print(f"transaction_features rows    : {transaction_features.count():,}")
print(f"transaction_features columns : {len(transaction_features.columns)}")
transaction_features.printSchema()

transaction_features rows    : 13,305,915
transaction_features columns : 19
root
 |-- id: long (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- amount_clean: double (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- is_weekend: boolean (nullable = false)
 |-- use_chip_encoded: integer (nullable = false)
 |-- merchant_state: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- error_encoded: integer (nullable = false)
 |-- is_fraud_label: integer (nullable = false)
 |-- credit_score: integer (nullable = true)
 |-- debt_to_income_ratio: double (nullable = true)
 |-- avg_spend_per_transaction: double (nullable = true)
 |-- error_rate: double (nullable = true)
 |-- pct_online_transactions: double (nullable = true)
 |-- max_account_tenure_days: integer (nullable = true)
 |-- amount_vs_customer_avg: double (nullable = true)
 |-- is_home_state: integer (nullable = false)
 |-- is_high_risk_hour: integer (nullable = false)



In [0]:
#write outputs to curated

customer_features.write.mode("overwrite")\
    .parquet(f"{CURATED}/customer_features/")
print("customer_features written")

transaction_features.write.mode("overwrite") \
    .parquet(f"{CURATED}/transaction_features/")
print("transaction_features written ✓")


customer_features written
transaction_features written ✓


In [0]:
df_check_cust = spark.read.parquet(f"{CURATED}/customer_features/")
df_check_txn   = spark.read.parquet(f"{CURATED}/transaction_features/")

print("=== Verification read-back ===")
print(f"customer_features: {df_check_cust.count():,} rows, " f"{len(df_check_cust.columns)} columns")
print(f"transaction_features : {df_check_txn.count():,} rows," f"{len(df_check_txn.columns)} columns")

=== Verification read-back ===
customer_features: 1,219 rows, 42 columns
transaction_features : 13,305,915 rows,19 columns
